In [2]:
%pip install requests>=2.31.0 python-dotenv tavily-python langchain langchain-core

Note: you may need to restart the kernel to use updated packages.


In [5]:
from langchain_core.tools import tool
from dotenv import load_dotenv
import os
from tavily import TavilyClient
from rich import print
from bs4 import BeautifulSoup
# trafilatura is installed in a separate notebook cell
import re

load_dotenv()

api_key = os.getenv("TAVILY_API_KEY")
if not api_key:
    raise RuntimeError("TAVILY_API_KEY is not set in the environment")

tavily = TavilyClient(api_key=api_key)

@tool
def web_search(query: str) -> str:
    """Search the web and return titles, URLs, and short snippets."""
    results = tavily.search(query=query, max_results=5)
    out = []

    for result in results.get("results", []):
        out.append(
            f"Title: {result.get('title', '')}\n"
            f"URL: {result.get('url', '')}\n"
            f"Snippet: {result.get('content', '')[:300]}"
        )

    return "\n----\n".join(out) or "No results found."


In [13]:
%pip install -q requests beautifulsoup4

Note: you may need to restart the kernel to use updated packages.


In [ ]:

import requests
from bs4 import BeautifulSoup
from urllib.parse import urlparse

@tool
def scrape_url(url: str) -> str:
    """
    Scrape a URL and return clean, readable text.

    Uses multiple extraction strategies:
    1. Trafilatura
    2. Readability
    3. BeautifulSoup fallback
    """

    # Validate URL
    parsed = urlparse(url)

    if parsed.scheme not in ("http", "https"):
        raise ValueError("Invalid URL. Use http:// or https://")

    headers = {
        "User-Agent": (
            "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
            "AppleWebKit/537.36 Chrome/140.0 Safari/537.36"
        )
    }

    try:
        response = requests.get(
            url,
            headers=headers,
            timeout=15
        )

        response.raise_for_status()

    except requests.RequestException as e:
        return f"Failed to fetch URL: {e}"

    html = response.text

    # ------------------------------------------------
    # Strategy 1: Trafilatura
    # ------------------------------------------------
    try:
        import trafilatura

        text = trafilatura.extract(
            html,
            include_links=False,
            include_images=False,
            include_tables=True
        )

        if text and len(text.strip()) > 100:
            return clean_text(text)

    except ImportError:
        pass
    except Exception:
        pass

    # ------------------------------------------------
    # Strategy 2: Readability
    # ------------------------------------------------
    try:
        from readability import Document

        doc = Document(html)
        main_html = doc.summary()

        soup = BeautifulSoup(main_html, "html.parser")
        text = soup.get_text("\n", strip=True)

        if text and len(text.strip()) > 100:
            return clean_text(text)

    except ImportError:
        pass
    except Exception:
        pass

    # ------------------------------------------------
    # Strategy 3: BeautifulSoup fallback
    # ------------------------------------------------
    try:
        soup = BeautifulSoup(html, "html.parser")

        # Remove unwanted elements
        for tag in soup([
            "script",
            "style",
            "noscript",
            "nav",
            "footer",
            "header",
            "aside",
            "form",
            "iframe",
            " 광고"
        ]):
            tag.decompose()

        # Prefer article/main content
        main = (
            soup.find("article")
            or soup.find("main")
            or soup.find("body")
        )

        if not main:
            return "No readable content found."

        text = main.get_text("\n", strip=True)

        return clean_text(text)

    except Exception as e:
        return f"Content extraction failed: {e}"


def clean_text(text: str) -> str:
    """Clean and normalize extracted text."""

    lines = []

    for line in text.splitlines():
        line = " ".join(line.split())

        if line and line not in lines:
            lines.append(line)

    return "\n".join(lines)


The important part of your project is the fallback architecture:
                 URL
                  ↓
             requests
                  ↓
        ┌─────────────────┐
        │ Try Trafilatura │
        └────────┬────────┘
                 │ fails
                 ↓
        ┌─────────────────┐
        │ Try Readability │
        └────────┬────────┘
                 │ fails
                 ↓
        ┌─────────────────┐
        │ BeautifulSoup   │
        │    fallback     │
        └────────┬────────┘
                 ↓
          Clean Text
                 ↓
             LLM / RAG